In [ ]:
# 1. Data Loading and Exploration
import pandas as pd
import numpy as np
from scipy import stats, signal
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11,5)

# set your path, e.g. "AAPL.csv"
CSV_PATH = "AAPL.csv"
df = pd.read_csv(CSV_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df["Year"] = df["Date"].dt.year

print("Nulls per column:\n", df.isna().sum())
print("\nDtypes:\n", df.dtypes)
print("\nDate range:", df["Date"].min().date(), "->", df["Date"].max().date())
# time series frequency (rough): median days between rows
print("Median delta days:", df["Date"].diff().dt.days.median())

# 2. Data Visualization
# 2.1 Close & Volume over time
fig, ax1 = plt.subplots()
ax1.plot(df["Date"], df["Close"], label="Close")
ax1.set_xlabel("Date"); ax1.set_ylabel("Close")
ax2 = ax1.twinx()
ax2.bar(df["Date"], df["Volume"], alpha=0.3)
ax2.set_ylabel("Volume")
ax1.set_title("AAPL Close and Volume")
fig.tight_layout(); plt.show()

# 2.2 Candlestick (requires mplfinance)
try:
    import mplfinance as mpf
    last = df.set_index("Date").tail(120)[["Open","High","Low","Close","Volume"]]
    mpf.plot(last, type="candle", volume=True, title="AAPL Candlestick (last 120 sessions)")
except Exception as e:
    print("Candlestick skipped (mplfinance not available):", e)

# 3. Statistical Analysis
cols = ["Open","High","Low","Close","Adj Close","Volume"]
existing = [c for c in cols if c in df.columns]
print("\nSummary stats (selected):\n", df[existing].describe()[["mean","50%","std"]].rename(columns={"50%":"median"}))

# moving average on Close (Pandas)
for w in (20, 50, 200):
    df[f"SMA_{w}"] = df["Close"].rolling(w).mean()

plt.plot(df["Date"], df["Close"], label="Close")
plt.plot(df["Date"], df["SMA_20"], label="SMA 20")
plt.plot(df["Date"], df["SMA_50"], label="SMA 50")
plt.title("Close with Moving Averages")
plt.legend(); plt.tight_layout(); plt.show()

# 4. Hypothesis Testing
# 4.1 t-test on average Close between two years with enough data
year_counts = df.groupby("Year").size().sort_index()
years = year_counts[year_counts > 50].index.tolist()
if len(years) >= 2:
    y1, y2 = years[0], years[-1]
    c1 = df.loc[df["Year"] == y1, "Close"].dropna()
    c2 = df.loc[df["Year"] == y2, "Close"].dropna()
    t_stat, p_val = stats.ttest_ind(c1, c2, equal_var=False)
    print(f"\nT-test Close means: {y1} vs {y2} -> t={t_stat:.3f}, p={p_val:.4g}")
else:
    print("\nT-test skipped (not enough distinct years).")

# 4.2 Daily returns distribution & normality
df["Return"] = df["Close"].pct_change()
rets = df["Return"].dropna()
k2, p_norm = stats.normaltest(rets)  # D’Agostino’s K^2
print(f"Daily returns normality test: K2={k2:.3f}, p={p_norm:.4g}")
plt.hist(rets, bins=50, density=True)
plt.title("AAPL Daily Returns Distribution"); plt.tight_layout(); plt.show()

# 5. Advanced Statistical Techniques (Bonus)
# 5.1 NumPy: SMA via np.convolve (same as rolling mean)
def sma_np(series, window):
    w = np.ones(window) / window
    out = np.convolve(series, w, mode="valid")
    return np.concatenate([np.full(window-1, np.nan), out])

df["SMA20_np"] = sma_np(df["Close"].to_numpy(), 20)

# 5.2 Correlations between moving averages of Close and Volume
if "Volume" in df.columns:
    df["VMA20"] = df["Volume"].rolling(20).mean()
    valid = df[["SMA_20","VMA20"]].dropna()
    corr = np.corrcoef(valid["SMA_20"], valid["VMA20"])[0,1]
    print("Corr(SMA_20 Close, VMA20 Volume):", round(corr, 4))

# 5.3 Signal processing (SciPy): Savitzky–Golay smoothing of Close
df["Close_savgol"] = signal.savgol_filter(df["Close"], window_length=21, polyorder=2, mode="interp")
plt.plot(df["Date"], df["Close"], label="Close", alpha=0.5)
plt.plot(df["Date"], df["Close_savgol"], label="Savitzky–Golay", linewidth=2)
plt.title("Close with Savitzky–Golay Smoothing"); plt.legend(); plt.tight_layout(); plt.show()

# 6. Summary and Insights (prints)
print("\nSummary:")
print("- Close mean:", df['Close'].mean().round(2))
print("- Close std:", df['Close'].std().round(2))
print("- SMA(20) vs SMA(50) latest:", df[['SMA_20','SMA_50']].tail(1).to_dict('records')[0])

# 7. Reflection (prints)
print("\nReflection:")
print("- Data gaps and corporate actions may affect stationarity and comparability across years.")
print("- Normality is usually rejected for raw daily returns; heavy tails are common.")
print("- Moving-average choices (window length) materially change signals; parameter tuning is critical.")
